# 06 Model Explainability with SHAP

This notebook is part of a reproducible machine learning project using the BRFSS 2015 diabetes health indicators dataset. All outputs are saved automatically to the `results` directory.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "code":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "code"))

from project_utils import *
set_reproducibility()

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")

Project root: /Users/guuuuu/Desktop/final0519
Dataset path: /Users/guuuuu/Desktop/final0519/diabetes_012_health_indicators_BRFSS2015.csv


## Purpose

This notebook applies SHAP explainability to a tree-based diabetes classifier. SHAP estimates how much each feature contributes to predictions. This is important for academic reporting because predictive accuracy alone does not explain which public health factors are associated with model decisions.

In [2]:
clean_df = clean_data(load_raw_data(), remove_duplicates=True)
X_train, X_test, y_train, y_test, features = split_data(clean_df)

model_path = MODELS_DIR / "tuned_lightgbm.joblib"
if model_path.exists():
    explain_model = joblib.load(model_path)
    print("Loaded tuned LightGBM for SHAP.")
else:
    print("Tuned LightGBM not found; training baseline LightGBM for SHAP.")
    explain_model = make_pipeline(base_estimators()["LightGBM"], features, use_smote=True)
    X_fit, y_fit = sample_for_training(X_train, y_train, max_rows=35000)
    explain_model.fit(X_fit, y_fit)
    joblib.dump(explain_model, MODELS_DIR / "lightgbm_for_shap.joblib")

shap_table = shap_analysis(explain_model, X_train, X_test, features)
display(shap_table.head(15))

Loaded tuned LightGBM for SHAP.
Computing SHAP values on a reproducible sample...


 39%|========            | 708/1800 [00:11<00:16]       

 43%|=========           | 781/1800 [00:12<00:15]       

 48%|==========          | 856/1800 [00:13<00:14]       

 52%|==========          | 930/1800 [00:14<00:13]       

...

 84%|=================   | 1521/1800 [00:22<00:04]       

 89%|==================  | 1595/1800 [00:23<00:02]       

 93%|=================== | 1670/1800 [00:24<00:01]       

 97%|=================== | 1738/1800 [00:25<00:00]       

,Feature,Mean_ABS_SHAP
13,GenHlth,0.190428
18,Age,0.173551
0,HighBP,0.130597
3,BMI,0.130016
1,HighChol,0.104683
17,Sex,0.055353
20,Income,0.049088
19,Education,0.040394
14,MentHlth,0.040388
10,HvyAlcoholConsump,0.040343


## Public Health Interpretation

The highest-ranked SHAP features identify variables the model uses most strongly when separating diabetes classes. In a public health context, features such as BMI, general health, high blood pressure, age, difficulty walking, cholesterol, and income may reflect metabolic risk, chronic disease burden, access to resources, and social determinants of health. SHAP does not prove causation; it explains the trained model's predictive behavior.